# Phase 4: Revenue & Category Analysis (SQL)

In this notebook, we use DuckDB to run high-performance SQL queries directly against our processed CSV data. This allows us to extract business-critical KPIs using standard SQL.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Connect to DuckDB

In [2]:
conn = duckdb.connect(database=':memory:')

# Read the CSV into a DuckDB relation
conn.execute("""
    CREATE VIEW orders AS 
    SELECT * FROM read_csv_auto('../data/processed/merged_olist_data.csv')
""")
print("View 'orders' created successfully.")

View 'orders' created successfully.


## 2. Top 10 Product Categories by Revenue

In [3]:
query = """
SELECT 
    product_category_name AS Category,
    SUM(price) AS Total_Revenue,
    COUNT(DISTINCT order_id) AS Total_Orders
FROM orders
WHERE product_category_name IS NOT NULL
GROUP BY Category
ORDER BY Total_Revenue DESC
LIMIT 10;
"""
top_categories = conn.execute(query).df()
top_categories

,Category,Total_Revenue,Total_Orders
0,health_beauty,1258681.34,8836
1,watches_gifts,1205005.68,5624
2,bed_bath_table,1036988.68,9417
3,sports_leisure,988048.97,7720
4,computers_accessories,911954.32,6689
5,furniture_decor,729762.49,6449
6,cool_stuff,635290.85,3632
7,housewares,632248.66,5884
8,auto,592720.11,3897
9,garden_tools,485256.46,3518


## 3. Month-over-Month Revenue Growth

In [4]:
query = """
WITH monthly_revenue AS (
    SELECT 
        DATE_TRUNC('month', CAST(order_purchase_timestamp AS TIMESTAMP)) AS Month,
        SUM(price) AS Revenue
    FROM orders
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY Month
)
SELECT 
    Month,
    Revenue,
    LAG(Revenue) OVER (ORDER BY Month) AS Previous_Month_Revenue,
    ((Revenue - LAG(Revenue) OVER (ORDER BY Month)) / LAG(Revenue) OVER (ORDER BY Month)) * 100 AS MoM_Growth_Pct
FROM monthly_revenue
ORDER BY Month;
"""
mom_growth = conn.execute(query).df()
mom_growth.dropna().head(10)

,Month,Revenue,Previous_Month_Revenue,MoM_Growth_Pct
1,2016-10-01,49507.66,267.36,1.841723e+04
2,2016-12-01,10.90,49507.66,-9.997798e+01
3,2017-01-01,120312.87,10.90,1.103688e+06
4,2017-02-01,247303.02,120312.87,1.055499e+02
5,2017-03-01,374344.30,247303.02,5.137069e+01
6,2017-04-01,359927.23,374344.30,-3.851286e+00
7,2017-05-01,506071.14,359927.23,4.060374e+01
8,2017-06-01,433038.60,506071.14,-1.443128e+01
9,2017-07-01,498031.48,433038.60,1.500857e+01
10,2017-08-01,573971.68,498031.48,1.524807e+01


## 4. Average Order Value (AOV) by State

In [5]:
query = """
WITH order_totals AS (
    SELECT 
        order_id,
        customer_state,
        SUM(price + freight_value) AS Order_Total
    FROM orders
    WHERE customer_state IS NOT NULL
    GROUP BY order_id, customer_state
)
SELECT 
    customer_state AS State,
    AVG(Order_Total) AS AOV,
    COUNT(order_id) AS Total_Orders
FROM order_totals
GROUP BY State
HAVING COUNT(order_id) > 1000
ORDER BY AOV DESC;
"""
aov_state = conn.execute(query).df()
aov_state

,State,AOV,Total_Orders
0,CE,207.691258,1336
1,PE,195.532579,1652
2,BA,182.104428,3380
3,GO,173.247100,2020
4,SC,168.940642,3637
5,RJ,166.876820,12852
6,DF,166.225619,2140
7,RS,163.075619,5466
8,MG,160.790150,11635
9,ES,160.396005,2033


## 5. Seller Performance Ranking

In [6]:
query = """
SELECT 
    seller_id,
    COUNT(DISTINCT order_id) AS Total_Orders_Fulfilled,
    SUM(price) AS Total_Revenue,
    AVG(delivery_delay_days) AS Avg_Delivery_Delay,
    AVG(review_score) AS Avg_Review_Score
FROM orders
WHERE seller_id IS NOT NULL
GROUP BY seller_id
HAVING COUNT(DISTINCT order_id) > 100
ORDER BY Avg_Review_Score DESC, Total_Orders_Fulfilled DESC
LIMIT 10;
"""
top_sellers = conn.execute(query).df()
top_sellers

,seller_id,Total_Orders_Fulfilled,Total_Revenue,Avg_Delivery_Delay,Avg_Review_Score
0,289cdb325fb7e7f891c38608bf9e0962,110,13544.95,-9.872724,4.579365
1,ac3508719a1d8f5b7614b798f70af136,103,12775.84,-17.372371,4.568627
2,12b9676b00f60f3b700e83af21824c0e,135,26774.00,-20.567279,4.537313
3,080102cd0a76b09e0dcf55fcacc60e05,124,5716.78,-11.131870,4.492308
4,66922902710d126a0e7d26b0e3805106,151,14362.30,-9.546910,4.451613
5,c3cfdc648177fdbbbb35635a37472c53,278,43048.18,-15.629606,4.447712
6,6cd68b3ed6d59aaa9fece558ad360c0a,149,11188.57,-13.641445,4.445860
7,5cf13accae3222c70a9cac40818ae839,149,14383.20,-13.769761,4.438710
8,fa40cc5b934574b62717c68f3d678b6d,307,16055.87,-10.687421,4.437870
9,edb1ef5e36e0c8cd84eb3c9b003e486d,166,79284.55,-12.843339,4.434286
